# Conversational Search Retrieval Augmented Generation

In this notebook you will implement the following steps:

- **Answer selection + evaluation**: Implement a *search-based* conversation framework evaluation framework to evaluate conversation topics made up of conversation turns.
- **Answer ranking**: Implement a *re-ranking method* to sort the initial search results. Evaluate the re-ranked results.
- **Conversation memory**: Implement a conversational context modeling method to keep track of the conversation state. 

Submission dates:
- **20 October**: first stage retrieval + conversation memory + evaluation
- **15 November**: re-ranking with LLM + evaluation
- **15 December**: answer generation + evaluation

## Test bed and conversation topics
The TREC CAST corpus (http://www.treccast.ai/) for Conversational Search is indexed in this cluster and available to be searched behind an OpenSearch API.

The queries and the relevance judgments are available through class `ConvSearchEvaluation`:

In [4]:
from re import search

from tornado.escape import json_encode

import Memory
import TRECCASTeval as trec
import numpy as np
import pprint
import Memory as Mem

Mem = Mem.Memory()


import numpy as np

pp = pprint.PrettyPrinter(indent=4)

test_bed = trec.ConvSearchEvaluation()

print()
print("========================================== Training conversations =====")
topics = {}
memory = {}
#memory = {} #Memória com os tops 1st Stage
for topic in test_bed.train_topics:
    conv_id = topic['number']

    if conv_id not in (1, 2, 4, 7, 15, 17,18,22,23,24,25,27,30):
        continue

    print()
    print(conv_id, "  ", topic['title'])

    for turn in topic['turn']:
        turn_id = turn['number']
        utterance = turn['raw_utterance']
        topic_turn_id = '%d_%d'% (conv_id, turn_id)
        
        print(topic_turn_id, utterance)
        topics[topic_turn_id] = utterance

print()
print("========================================== Test conversations =====")
for topic in test_bed.test_topics:
    conv_id = topic['number']

    if conv_id not in (31, 32, 33, 34, 37, 40, 49, 50, 54, 56, 58, 59, 61, 67, 68, 69, 75, 77, 78, 79):
        continue

    print()
    print(conv_id, "  ", topic['title'])

    for turn in topic['turn']:
        turn_id = turn['number']
        utterance = turn['raw_utterance']
        topic_turn_id = '%d_%d'% (conv_id, turn_id)
        
        print(topic_turn_id, utterance)
        topics[topic_turn_id] = utterance
        
        
        
print()
print("TESTE ======================================== Topics ========")
print(topics)





========================================== Training conversations =====

1    Career choice for Nursing and Physician's Assistant
1_1 What is a physician's assistant?
1_2 What are the educational requirements required to become one?
1_3 What does it cost?
1_4 What's the average starting salary in the UK?
1_5 What about in the US?
1_6 What school subjects are needed to become a registered nurse?
1_7 What is the PA average salary vs an RN?
1_8 What the difference between a PA and a nurse practitioner?
1_9 Do NPs or PAs make more?
1_10 Is a PA above a NP?
1_11 What is the fastest way to become a NP?
1_12 How much longer does it take to become a doctor after being an NP?

2    Goat breeds
2_1 What are the main breeds of goat?
2_2 Tell me about boer goats.
2_3 What breed is good for meat?
2_4 Are angora goats good for it?
2_5 What about boer goats?
2_6 What are pygmies used for?
2_7 What is the best for fiber production?
2_8 How long do Angora goats live?
2_9 Can you milk them?
2_10 How ma

In [2]:
test_bed.test_relevance_judgments



,topic_turn_id,dummy,docid,rel
0,31_1,Q0,CAR_116d829c4c800c2fc70f11692fec5e8c7e975250,0
1,31_1,Q0,CAR_1463f964653c5c9f614a0a88d26b175e4a8120f1,1
2,31_1,Q0,CAR_172e16e89ea3d5546e53384a27c3be299bcfe968,2
3,31_1,Q0,CAR_1c93ef499a0c2856c4a857b0cb4720c380dda476,0
4,31_1,Q0,CAR_2174ad0aa50712ff24035c23f59a3c2b43267650,3
...,...,...,...,...
29345,79_9,Q0,MARCO_8795229,0
29346,79_9,Q0,MARCO_8795231,0
29347,79_9,Q0,MARCO_8795233,0
29348,79_9,Q0,MARCO_8795236,0


## OpenSearch

In [5]:
import OpenSearchSimpleAPI as osearch
import pprint as pp
opensearch = osearch.OSsimpleAPI()



Search example:

In [4]:
results = opensearch.search_body(topics['33_1'], numDocs = 100)
print(results)


# #1ª Tentativa para o 1st Stage
# for topic in topics:
#     print("================="+ topic +"================")
#     results = opensearch.search_body(topics[topic], numDocs = 10)
#     print(results[:3]["_id"])
#     
#     top = [results[:3]["_id"]]
#     title = topics[topic]
#     print(title)
#     memory[topic] = (title, top)
#     
# print(memory)


   _index _type                                           _id     _score  \
0    kwiz  _doc                                   MARCO_34196  21.662935   
1    kwiz  _doc  CAR_c370ef5df77de117ff7d02c4b64b52f5bae9abc9  21.348211   
2    kwiz  _doc                                 MARCO_6213095  19.961092   
3    kwiz  _doc  CAR_77e4e91100d10a01d66980d12c1911b2480188f2  19.919603   
4    kwiz  _doc  CAR_bc696d8a1aa65b9ad1457552248018e22bda9a46  18.935474   
..    ...   ...                                           ...        ...   
95   kwiz  _doc  CAR_841790aa4b0e3819aee7e0c40be3eca5b0b9d219  15.131712   
96   kwiz  _doc                                 MARCO_5162502  15.120180   
97   kwiz  _doc                                 MARCO_2763742  15.029955   
98   kwiz  _doc  CAR_98f7a97ae8449c31ae3b32c4dfce28ffdaa24c1b  15.010900   
99   kwiz  _doc                                 MARCO_6134966  15.007110   

                                     _source.contents  \
0   The holidays are prime tim

In [17]:
opensearch.doc_term_vectors(doc_id='MARCO_34196')

(23596,
 1405893,
 2299289,
 {'a': [4, 17146, 48957],
  'about': [2, 2688, 3958],
  'alex': [1, 36, 79],
  'and': [2, 19644, 68602],
  'are': [1, 8821, 17501],
  'at': [1, 4726, 7343],
  'been': [1, 2550, 3343],
  'best': [1, 990, 1687],
  'bits': [2, 26, 30],
  'bitsâ': [1, 1, 1],
  'can': [1, 4720, 7495],
  'catches': [1, 12, 16],
  'children': [1, 362, 559],
  'chris': [1, 63, 77],
  'dadâ': [1, 1, 1],
  'dark': [1, 333, 454],
  'else': [1, 104, 114],
  'everyone': [1, 106, 130],
  'family': [1, 555, 775],
  'for': [2, 9865, 18393],
  'friend': [1, 130, 156],
  'from': [1, 7167, 11274],
  'funny': [1, 26, 28],
  'gory': [1, 2, 2],
  'hand': [1, 311, 415],
  'has': [2, 3998, 5765],
  'having': [1, 503, 579],
  'himself': [1, 246, 289],
  'his': [2, 2353, 5180],
  'holiday': [1, 46, 60],
  'holidays': [2, 19, 21],
  'in': [2, 17309, 52844],
  'into': [1, 2473, 3267],
  'invite': [1, 5, 5],
  'just': [1, 909, 1128],
  'kids': [1, 88, 135],
  'leave': [1, 131, 153],
  'little': [1, 436,

In [13]:
opensearch.termvectros_JSON(doc_id='MARCO_34196')

{'_index': 'kwiz',
 '_type': '_doc',
 '_id': 'MARCO_34196',
 '_version': 1,
 'found': True,
 'took': 0,
 'term_vectors': {'contents': {'field_statistics': {'sum_doc_freq': 1405893,
    'doc_count': 23596,
    'sum_ttf': 2299289},
   'terms': {'a': {'doc_freq': 17146, 'ttf': 48957, 'term_freq': 4},
    'about': {'doc_freq': 2688, 'ttf': 3958, 'term_freq': 2},
    'alex': {'doc_freq': 36, 'ttf': 79, 'term_freq': 1},
    'and': {'doc_freq': 19644, 'ttf': 68602, 'term_freq': 2},
    'are': {'doc_freq': 8821, 'ttf': 17501, 'term_freq': 1},
    'at': {'doc_freq': 4726, 'ttf': 7343, 'term_freq': 1},
    'been': {'doc_freq': 2550, 'ttf': 3343, 'term_freq': 1},
    'best': {'doc_freq': 990, 'ttf': 1687, 'term_freq': 1},
    'bits': {'doc_freq': 26, 'ttf': 30, 'term_freq': 2},
    'bitsâ': {'doc_freq': 1, 'ttf': 1, 'term_freq': 1},
    'can': {'doc_freq': 4720, 'ttf': 7495, 'term_freq': 1},
    'catches': {'doc_freq': 12, 'ttf': 16, 'term_freq': 1},
    'children': {'doc_freq': 362, 'ttf': 559, 

In [8]:
opensearch.get_doc_body('MARCO_34196')

'The holidays are prime time for family storytelling. When youâ\x80\x99re putting up the tree or having your holiday meal, share a story with your children about past holidays. Leave in the funny bits, the sad bits, the gory and smelly bitsâ\x80\x93kids can tell when a story has been sanitized for their protection.Then invite everyone else to tell a story too.Dad, tell me a story from when you were little. Tell me the story about the time you met your best friend Chris at school.â\x80\x9d Six-year-old Alex, who has just started school himself, snuggles into his pillow and catches his dadâ\x80\x99s hand in the dark.'

In [9]:
example_doc = 'The film only adapts the first half of the book, and consequently does not convey the message of the title as it was portrayed in the novel. The second half of the book would subsequently be used as the rough basis for the second film, The NeverEnding Story II: The Next Chapter.The third film, The NeverEnding Story III: Escape From Fantasia, features a completely original plot.he second half of the book would subsequently be used as the rough basis for the second film, The NeverEnding Story II: The Next Chapter.'
opensearch.query_terms(example_doc,'standard')

' the film only adapts the first half of the book and consequently does not convey the message of the title as it was portrayed in the novel the second half of the book would subsequently be used as the rough basis for the second film the neverending story ii the next chapter.the third film the neverending story iii escape from fantasia features a completely original plot.he second half of the book would subsequently be used as the rough basis for the second film the neverending story ii the next chapter'

In [10]:
opensearch.analyzer(analyzer="standard", query=example_doc)

{'tokens': [{'token': 'the',
   'start_offset': 0,
   'end_offset': 3,
   'type': '<ALPHANUM>',
   'position': 0},
  {'token': 'film',
   'start_offset': 4,
   'end_offset': 8,
   'type': '<ALPHANUM>',
   'position': 1},
  {'token': 'only',
   'start_offset': 9,
   'end_offset': 13,
   'type': '<ALPHANUM>',
   'position': 2},
  {'token': 'adapts',
   'start_offset': 14,
   'end_offset': 20,
   'type': '<ALPHANUM>',
   'position': 3},
  {'token': 'the',
   'start_offset': 21,
   'end_offset': 24,
   'type': '<ALPHANUM>',
   'position': 4},
  {'token': 'first',
   'start_offset': 25,
   'end_offset': 30,
   'type': '<ALPHANUM>',
   'position': 5},
  {'token': 'half',
   'start_offset': 31,
   'end_offset': 35,
   'type': '<ALPHANUM>',
   'position': 6},
  {'token': 'of',
   'start_offset': 36,
   'end_offset': 38,
   'type': '<ALPHANUM>',
   'position': 7},
  {'token': 'the',
   'start_offset': 39,
   'end_offset': 42,
   'type': '<ALPHANUM>',
   'position': 8},
  {'token': 'book',
   's

In [4]:
#Ranking with raw utterances
import pandas as pd

for topic in topics:
    res = opensearch.search_body(topics[topic], numDocs = 100)
    output = pd.DataFrame({
    'ID': res['_id'],
    'Score': res['_score']
    })
    print(str(topic) + ' query: ' + topics[topic])
    print(output)
    print()


1_1 query: What is a physician's assistant?
                                              ID      Score
0                                  MARCO_3951096  22.857110
1                                   MARCO_849267  21.770878
2                                   MARCO_920443  21.770878
3                                  MARCO_2307794  21.652912
4                                  MARCO_4389107  14.715845
..                                           ...        ...
95  CAR_e547a542463cd03cadad9a8c2b879adaadd79498   6.848598
96                                 MARCO_3984224   6.829070
97                                 MARCO_2486726   6.754774
98                                 MARCO_5417120   6.709174
99                                 MARCO_5690791   6.642787

[100 rows x 2 columns]

1_2 query: What are the educational requirements required to become one?
                                              ID      Score
0                                  MARCO_7225587  22.910360
1                 

In [15]:
#Conversation History
import Memory as Mem

Mem = Mem.Memory()


memory = Mem.storeMem(topics= topics, nDocs=100, nTop=3, opensearch=opensearch)
print(memory)


=================1_1================
0    MARCO_3951096
1     MARCO_849267
2     MARCO_920443
Name: _id, dtype: object
TOP -->['MARCO_3951096', 'MARCO_849267', 'MARCO_920443']
What is a physician's assistant?
=================1_2================
0    MARCO_7225587
1    MARCO_7342284
2     MARCO_355203
Name: _id, dtype: object
TOP -->['MARCO_7225587', 'MARCO_7342284', 'MARCO_355203']
What are the educational requirements required to become one?
=================1_3================
0    MARCO_7195601
1    MARCO_1513207
2    MARCO_1575196
Name: _id, dtype: object
TOP -->['MARCO_7195601', 'MARCO_1513207', 'MARCO_1575196']
What does it cost?
=================1_4================
0    MARCO_4881319
1    MARCO_6249674
2    MARCO_4881318
Name: _id, dtype: object
TOP -->['MARCO_4881319', 'MARCO_6249674', 'MARCO_4881318']
What's the average starting salary in the UK?
=================1_5================
0    MARCO_3340202
1    MARCO_4077346
2     MARCO_146883
Name: _id, dtype: object
TOP -->['MAR

In [ ]:
#Ranking with conversation history
#TODO


In [6]:
#Re-Ranking
import ReRank as RR
import math

# rr = RR.ReRank()

# rerank = rr.reRank(topics = topics, nDocs = 100, opensearch = opensearch, memory = memory)

# for topic, (query, doc_ids) in  memory.items():

#USAR ISTO PARA FAZER O CICLO POR TODOS 
# for topic in topics:
#     res = opensearch.search_body(topics[topic], numDocs = 100)
    
res = opensearch.search_body(topics['33_1'], numDocs = 100)
print(res)

queryTerms = opensearch.query_terms(analyzer="standard", query=topics['33_1']).split()
print(queryTerms)

topDocs =  res[:3]['_id'].tolist()
    
col_doc_freq, col_sum_doc_freq, col_sum_ttf, col_docs_term_vectors = opensearch.multi_doc_term_vectors(doc_ids = topDocs)

print(
    f'================================== Doc_freq: {col_doc_freq} Sum_doc_freq: {col_sum_doc_freq} Sum_ttf: {col_sum_ttf}')
    
for doc, terms in col_docs_term_vectors.items():
    print(doc)
    print(terms)
    d = sum(terms[term][0] for term in terms)
    miu = 0.1 * d
    
    res = 0
    for term in queryTerms:
        # print('TERM: ' + str(term))
    
        if term not in terms:
            print('NÃO ESTÁ -------------> ' + str(term))
            
            found = False
            for col_docs, col_terms in col_docs_term_vectors.items():
                if term in col_terms:
                    found = True
                    ttf = col_terms[term][2]
                    print(col_terms[term])
                    print('ttf' + str(ttf))
                    
                    term_freq = 0
                    prob = ttf/col_sum_ttf
                    
                    numerator = term_freq + miu * prob
                    denominator = miu + d
                    
                    func = numerator/denominator
                    
                    res += math.log(func)
                    
                    break
            
            if not found:
                print('NÃO ESTÁ GERAL -------------> ' + str(term))
        
        else: 
            print('ESTÁ: ' + str(term))
            print(terms[term])
            
            term_freq = terms[term][0]
            prob = terms[term][2]/col_sum_ttf
            
            numerator = term_freq + miu * prob
            denominator = miu + d
            
            func = numerator/denominator
            
            res += math.log(func)
            
    print('SCORE ----->' + str(math.exp(res)))   
            
               
         
            # if not any(term in terms for terms in col_docs_term_vectors.values()):
            #     print('NÃO ESTÁ GERAL -------------> ' + str(term))
            
            # for doc_id, col_terms in col_docs_term_vectors.items():

            #     if term not in col_terms:
            #         print('NÃO ESTÁ GERAL -------------> ' + str(term))
            
            
            
        
        
        
        
    
    
        
     
    # for doc in topDocs:
    #     termVec = opensearch.termvectros_JSON(doc_id=doc)
    #     print('DOC: ' + str(doc))
    #     print(
    #     f'================================== Doc_freq: {doc_freq} Sum_doc_freq: {sum_doc_freq} Sum_ttf: {sum_ttf} Docs_term_vectors: {docs_term_vectors}')
    #     
    #     print(termVec)
        
        
        




   _index _type                                           _id     _score  \
0    kwiz  _doc                                   MARCO_34196  21.662935   
1    kwiz  _doc  CAR_c370ef5df77de117ff7d02c4b64b52f5bae9abc9  21.348211   
2    kwiz  _doc                                 MARCO_6213095  19.961092   
3    kwiz  _doc  CAR_77e4e91100d10a01d66980d12c1911b2480188f2  19.919603   
4    kwiz  _doc  CAR_bc696d8a1aa65b9ad1457552248018e22bda9a46  18.935474   
..    ...   ...                                           ...        ...   
95   kwiz  _doc  CAR_841790aa4b0e3819aee7e0c40be3eca5b0b9d219  15.131712   
96   kwiz  _doc                                 MARCO_5162502  15.120180   
97   kwiz  _doc                                 MARCO_2763742  15.029955   
98   kwiz  _doc  CAR_98f7a97ae8449c31ae3b32c4dfce28ffdaa24c1b  15.010900   
99   kwiz  _doc                                 MARCO_6134966  15.007110   

                                     _source.contents  \
0   The holidays are prime tim

In [ ]:
#Evaluation
#TODO

In [35]:
import math

#LMD

search = opensearch.search_body(topics['33_1'], numDocs = 100)
print(search['_id'][1]) 
termVec = opensearch.termvectros_JSON(doc_id=search['_id'][1])
print(termVec)  

sum_ttf = termVec['term_vectors']['contents']['field_statistics']['sum_ttf']
print('=========== sum_ttf')
print(sum_ttf)

d2 =  termVec['term_vectors']['contents']['field_statistics']['doc_count']
print('=========== d2: ')
print(d2)

d3 = termVec['term_vectors']['contents']['field_statistics']['sum_ttf']/termVec['term_vectors']['contents']['field_statistics']['doc_count']
print('=========== d3: ')
print(d3)



terms = termVec['term_vectors']['contents']['terms']
 
    
d1 = sum(terms[term]['term_freq'] for term in terms)
print('=========== d1: ')
print(d1)



miu = 0.1 * d3
print('=========== miu com d3: ')
print(miu)

miu2 = 0.1 * d1
print('=========== miu com d1: ')
print(miu2)

miu4 = 0.1 * d2

res = 1

denominator2 = d2 + miu4
denominator = d1 + miu2


for term in terms:
    
    print('Term : ' + term)
    
    term_freq = terms[term]['term_freq']
    
    prob = terms[term]['ttf']/ sum_ttf
    print('Probability : ' + str(prob))
    
    funcUp = term_freq + miu2 * prob
    print('Upside: ' + str(funcUp) + ' Downside: ' + str(denominator))
    # funcDown = d1 + miu3
    # print(funcDown)
    # func = (terms[term]['term_freq'] + (miu2 * (terms[term]['ttf']/sum_ttf))) / (miu2 + d1)
    func = funcUp/denominator
    print(func)
    # res = res + math.log(func)
    res *= func
        
print(res) 
# print(math.exp(res))


d8 = termVec['term_vectors']['contents']['field_statistics']['doc_count']

miu8 = 0.1 * (sum_ttf/d8)


res2 = 1
for term in terms:
    term_freq = terms[term]['term_freq']

    up = term_freq + miu8 * (terms[term]['ttf']/ sum_ttf)
    down = miu8 + d8
    
    func = up/down
    
    res2 *= func
    
print("============================================")
print(res2)









CAR_c370ef5df77de117ff7d02c4b64b52f5bae9abc9
{'_index': 'kwiz', '_type': '_doc', '_id': 'CAR_c370ef5df77de117ff7d02c4b64b52f5bae9abc9', '_version': 1, 'found': True, 'took': 0, 'term_vectors': {'contents': {'field_statistics': {'sum_doc_freq': 1405893, 'doc_count': 23596, 'sum_ttf': 2299289}, 'terms': {'1979': {'doc_freq': 86, 'ttf': 96, 'term_freq': 1}, '1994': {'doc_freq': 114, 'ttf': 130, 'term_freq': 1}, 'a': {'doc_freq': 17146, 'ttf': 48957, 'term_freq': 2}, 'also': {'doc_freq': 4940, 'ttf': 6628, 'term_freq': 1}, 'and': {'doc_freq': 19644, 'ttf': 68602, 'term_freq': 2}, 'as': {'doc_freq': 9879, 'ttf': 19825, 'term_freq': 3}, 'bastian': {'doc_freq': 29, 'ttf': 55, 'term_freq': 1}, 'black': {'doc_freq': 465, 'ttf': 695, 'term_freq': 1}, 'bully': {'doc_freq': 6, 'ttf': 9, 'term_freq': 1}, 'but': {'doc_freq': 3822, 'ttf': 5068, 'term_freq': 1}, 'bux': {'doc_freq': 15, 'ttf': 15, 'term_freq': 1}, 'chapter': {'doc_freq': 99, 'ttf': 144, 'term_freq': 1}, 'character': {'doc_freq': 445, '